<a target="_blank" href="https://colab.research.google.com/github/adamya-singh/machine-learning-exercises/blob/master/neuroscope-neel-nanda/Interactive_Neuroscope.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Interactive Neuroscope

*This is an interactive accompaniment to [neuroscope.io](https://neuroscope.io) and to the [studying learned language features post](https://www.alignmentforum.org/posts/Qup9gorqpd9qKAEav/200-cop-in-mi-studying-learned-features-in-language-models) in [200 Concrete Open Problems in Mechanistic Interpretability](https://neelnanda.io/concrete-open-problems)*

There's a surprisingly rich ecosystem of easy ways to create interactive graphics, especially for ML systems. If you're trying to do mechanistic interpretability, the ability to do web dev and to both visualize data and interact with it seems high value!

This version swaps the original dense GPT-2 neuron viewer for an OLMoE expert-routing viewer. The goal is still to show what quickly hacking together a custom visualisation can look like, while keeping the code simple enough to edit in a notebook.

Note that you'll need to run the code yourself to get the interactive interface, so the cell at the bottom will be blank at first!

To emphasise - the point of this notebook is to be a rough proof of concept that just about works, *not* to be the well executed ideal of interactively studying expert routing.

## Setup

In [1]:
# NBVAL_IGNORE_OUTPUT
# Janky code to do different setup when run in a Colab notebook vs VSCode
import os

DEVELOPMENT_MODE = True
IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
try:
    import google.colab

    IN_COLAB = True
    print("Running as a Colab notebook")
except:
    IN_COLAB = False
    print("Running as a Jupyter notebook - intended for development only!")
    from IPython import get_ipython

    ipython = get_ipython()
    # Code to automatically update notebook imports as they are edited without restarting the kernel
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

if IN_COLAB or IN_GITHUB:
    %pip install "transformers>=4.57.0"
    %pip install torch
    %pip install datasets
    %pip install gradio
    %pip install sentencepiece


Running as a Colab notebook


In [2]:
# NBVAL_IGNORE_OUTPUT
import html

import pandas as pd
import gradio as gr
import torch
from datasets import load_dataset
from IPython.display import HTML, display
from transformers import AutoTokenizer, OlmoeForCausalLM


## Loading DBPedia14 Title Groups

We load the `dbpedia_14` dataset, keep only article titles, and sample balanced groups from user-selected topic labels before running model inference.

The helpers below accept topic names or topic ids, use the canonical label names from the dataset, and return a compact table we can feed into the routing collection step.

In [3]:
def _normalize_dbpedia_topics(selected_topics, label_names):
    if isinstance(selected_topics, (str, int)):
        selected_topics = [selected_topics]

    if not selected_topics:
        raise ValueError("selected_topics must contain at least one topic name or id.")

    valid_topics_text = ", ".join(
        f"{label_id}:{topic_name}" for label_id, topic_name in enumerate(label_names)
    )
    topic_name_to_id = {topic_name.lower(): label_id for label_id, topic_name in enumerate(label_names)}

    normalized_label_ids = []
    seen_label_ids = set()
    for topic in selected_topics:
        if isinstance(topic, str):
            stripped_topic = topic.strip()
            if stripped_topic.isdigit():
                label_id = int(stripped_topic)
            else:
                label_id = topic_name_to_id.get(stripped_topic.lower())
                if label_id is None:
                    raise ValueError(
                        f"Unknown topic '{topic}'. Valid topics are: {valid_topics_text}"
                    )
        else:
            label_id = int(topic)

        if not 0 <= label_id < len(label_names):
            raise ValueError(
                f"Unknown topic id {label_id}. Valid topics are: {valid_topics_text}"
            )

        if label_id not in seen_label_ids:
            normalized_label_ids.append(label_id)
            seen_label_ids.add(label_id)

    return normalized_label_ids


def load_dbpedia_title_groups(selected_topics, split="test", max_titles_per_topic=100, seed=0):
    if max_titles_per_topic is not None:
        max_titles_per_topic = int(max_titles_per_topic)
        if max_titles_per_topic <= 0:
            raise ValueError("max_titles_per_topic must be positive or None.")

    dataset = load_dataset("dbpedia_14", split=split)
    label_names = dataset.features["label"].names
    selected_label_ids = _normalize_dbpedia_topics(selected_topics, label_names)

    dataset_df = dataset.to_pandas().reset_index().rename(columns={"index": "original_index"})
    filtered_df = dataset_df.loc[
        dataset_df["label"].isin(selected_label_ids), ["original_index", "label", "title"]
    ].copy()
    filtered_df["topic_name"] = filtered_df["label"].map(lambda label_id: label_names[int(label_id)])

    sampled_groups = []
    for label_id in selected_label_ids:
        topic_df = filtered_df.loc[filtered_df["label"] == label_id].copy()
        if max_titles_per_topic is None:
            topic_df = topic_df.sort_values("original_index", kind="stable").reset_index(drop=True)
        else:
            topic_df = topic_df.sample(
                n=min(max_titles_per_topic, len(topic_df)),
                random_state=seed,
                replace=False,
            ).reset_index(drop=True)
        topic_df["sample_order"] = range(len(topic_df))
        sampled_groups.append(topic_df)

    result_columns = ["example_id", "dataset_split", "label_id", "topic_name", "title"]
    if not sampled_groups:
        return pd.DataFrame(columns=result_columns)

    result_df = pd.concat(sampled_groups, ignore_index=True)
    result_df = result_df.rename(columns={"label": "label_id"})
    result_df["dataset_split"] = split
    result_df["example_id"] = result_df["dataset_split"] + ":" + result_df["original_index"].astype(str)
    result_df = result_df.sort_values(["label_id", "sample_order"], kind="stable").reset_index(drop=True)

    return result_df[result_columns]


def show_topic_group_counts(df):
    required_columns = {"label_id", "topic_name"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    return (
        df.groupby(["label_id", "topic_name"], sort=True)
        .size()
        .rename("title_count")
        .reset_index()
    )


In [4]:
selected_topics = ["Company", "Artist"]
split = "test"
max_titles_per_topic = 100
seed = 0

title_groups_df = load_dbpedia_title_groups(
    selected_topics=selected_topics,
    split=split,
    max_titles_per_topic=max_titles_per_topic,
    seed=seed,
)
show_topic_group_counts(title_groups_df)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dbpedia_14/train-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

dbpedia_14/test-00000-of-00001.parquet:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

,label_id,topic_name,title_count
0,0,Company,100
1,2,Artist,100


## Extracting Router Probabilities

We first write some code to extract OLMoE router probabilities for all experts at a given layer, for a given text.

In [5]:
# NBVAL_IGNORE_OUTPUT
model_name = "allenai/OLMoE-1B-7B-0924"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = OlmoeForCausalLM.from_pretrained(model_name, dtype="auto").to(device)
model.eval()
print(f"Loaded {model_name} on {device}")


config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

Loaded allenai/OLMoE-1B-7B-0924 on cuda


In [6]:
def _validate_layers(layers):
    num_layers = model.config.num_hidden_layers
    if layers is None:
        return list(range(num_layers))

    normalized_layers = []
    for layer in layers:
        layer = int(layer)
        if not 0 <= layer < num_layers:
            raise ValueError(f"Layer must be between 0 and {num_layers - 1}.")
        normalized_layers.append(layer)
    return normalized_layers


def _prepare_model_inputs(text):
    inputs = tokenizer(text, return_tensors="pt")
    return {key: value.to(device) for key, value in inputs.items()}


def _get_router_outputs(text):
    inputs = _prepare_model_inputs(text)
    with torch.no_grad():
        outputs = model(**inputs, output_router_logits=True)

    router_logits = outputs.router_logits
    if router_logits is None:
        raise ValueError("Model did not return router logits.")

    input_ids = inputs["input_ids"][0].detach().cpu().tolist()
    str_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    return inputs, router_logits, str_tokens


def _get_layer_router_probs(router_logits, input_shape, layer):
    layer_router_logits = router_logits[layer]
    if layer_router_logits.dim() == 2:
        batch_size, seq_len = input_shape
        layer_router_logits = layer_router_logits.view(batch_size, seq_len, -1)
    elif layer_router_logits.dim() != 3:
        raise ValueError(
            f"Unexpected router logits shape for layer {layer}: {tuple(layer_router_logits.shape)}"
        )

    return layer_router_logits[0].detach().to(torch.float32).cpu()


def collect_title_routing(df, layers=None):
    required_columns = {"example_id", "dataset_split", "label_id", "topic_name", "title"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    layer_indices = _validate_layers(layers)
    top_k = int(model.config.num_experts_per_tok)
    result_columns = [
        "example_id",
        "dataset_split",
        "label_id",
        "topic_name",
        "title",
        "layer",
        "token_position",
        "token_text",
        "selected_expert_ids",
        "selected_expert_scores",
    ]
    if df.empty:
        return pd.DataFrame(columns=result_columns)

    records = []
    for row in df.itertuples(index=False):
        inputs, router_logits, str_tokens = _get_router_outputs(row.title)
        for layer in layer_indices:
            layer_router_probs = _get_layer_router_probs(router_logits, inputs["input_ids"].shape, layer)
            selected_scores, selected_ids = torch.topk(layer_router_probs, k=top_k, dim=-1)
            for token_position, token_text in enumerate(str_tokens):
                records.append(
                    {
                        "example_id": row.example_id,
                        "dataset_split": row.dataset_split,
                        "label_id": row.label_id,
                        "topic_name": row.topic_name,
                        "title": row.title,
                        "layer": layer,
                        "token_position": token_position,
                        "token_text": token_text,
                        "selected_expert_ids": selected_ids[token_position].tolist(),
                        "selected_expert_scores": selected_scores[token_position].tolist(),
                    }
                )

    return pd.DataFrame.from_records(records, columns=result_columns)


def get_router_probs(text, layer):
    if layer is None:
        raise ValueError("Please select a layer.")

    normalized_layer = _validate_layers([layer])[0]
    inputs, router_logits, str_tokens = _get_router_outputs(text)
    router_probs = _get_layer_router_probs(router_logits, inputs["input_ids"].shape, normalized_layer)
    return router_probs.numpy(), str_tokens


We can run this function and verify that it gives a token-by-expert matrix of router probabilities.

In [7]:
default_layer = 7
default_text = "Sunday"
print(tokenizer.convert_ids_to_tokens(tokenizer(default_text, return_tensors="pt")["input_ids"][0]))


['Sunday']


In [8]:
# NBVAL_IGNORE_OUTPUT
router_probs, tokens = get_router_probs(default_text, default_layer)
print(router_probs.shape)
print(router_probs[0])
print(router_probs.sum(axis=1))


(1, 64)
[0.01186247 0.02435805 0.04031682 0.00676399 0.02132862 0.01063605
 0.00428269 0.00939087 0.00520643 0.00953875 0.00848394 0.00489098
 0.03502786 0.02398042 0.01051214 0.00468528 0.00472202 0.01628947
 0.00261796 0.0125691  0.00732791 0.10135648 0.01122568 0.01296176
 0.02183442 0.00283069 0.01350444 0.02407427 0.01010449 0.00681704
 0.00706096 0.02562693 0.00916437 0.03290563 0.01170997 0.02814567
 0.00970791 0.03395016 0.00655588 0.02949641 0.01148138 0.00807964
 0.0265439  0.01134779 0.00418348 0.06751797 0.0113272  0.01545271
 0.01168427 0.02760129 0.00658154 0.00623128 0.01491889 0.00289781
 0.0059692  0.00653032 0.01066205 0.00457674 0.02847745 0.0072851
 0.0031089  0.01379772 0.0083851  0.01153335]
[1.]


## Collecting Expert Choices Across Title Groups

We can now run the sampled DBPedia14 title groups through OLMoE and record the top-k selected experts for every token at every requested layer.

Set `layers = None` to collect all routed layers, or pass a smaller list such as `[0, 1]` for a quicker smoke test.

In [9]:
# NBVAL_IGNORE_OUTPUT
layers = None
title_routing_df = collect_title_routing(title_groups_df, layers=layers)
title_routing_df.head()


,example_id,dataset_split,label_id,topic_name,title,layer,token_position,token_text,selected_expert_ids,selected_expert_scores
0,test:398,test,0,Company,Sepura,0,0,Sep,"[57, 46, 24, 45, 47, 17, 22, 29]","[0.16334636509418488, 0.09677951782941818, 0.0..."
1,test:398,test,0,Company,Sepura,0,1,ura,"[47, 24, 45, 39, 63, 29, 52, 57]","[0.14502526819705963, 0.09363535046577454, 0.0..."
2,test:398,test,0,Company,Sepura,1,0,Sep,"[18, 11, 48, 56, 24, 45, 15, 25]","[0.09709381312131882, 0.09050138294696808, 0.0..."
3,test:398,test,0,Company,Sepura,1,1,ura,"[33, 49, 6, 43, 48, 56, 27, 11]","[0.12027820199728012, 0.07888036966323853, 0.0..."
4,test:398,test,0,Company,Sepura,2,0,Sep,"[30, 34, 60, 24, 59, 40, 45, 17]","[0.8593452572822571, 0.006997996941208839, 0.0..."


## Visualizing Router Probabilities

We now visualize router probabilities for all experts at one selected layer. The display is a simple token-by-expert heatmap built in raw HTML so it stays easy to hack on inside the notebook.

Each column is a token, each row is an expert, and each cell color shows that expert's routing probability for that token.

In [10]:
# This is some CSS to display a compact token-by-expert heatmap.
style_string = """<style>
    table.router-heatmap {
        border-collapse: collapse;
        font-family: monospace;
        font-size: 12px;
    }
    table.router-heatmap th, table.router-heatmap td {
        border: 1px solid rgb(123, 123, 123);
        padding: 4px;
        text-align: center;
        min-width: 48px;
    }
    table.router-heatmap th.token-header {
        writing-mode: vertical-rl;
        transform: rotate(180deg);
        white-space: pre-wrap;
    }
    table.router-heatmap th.expert-label {
        position: sticky;
        left: 0;
        background: white;
    }
    div.router-scroll {
        overflow-x: auto;
        max-width: 100%;
    }
    </style>"""


def calculate_color(val, max_val, min_val):
    # Normalize to [0, 1] and interpolate between slightly off-white and red.
    denom = max(max_val - min_val, 1e-8)
    normalized_val = min(max((val - min_val) / denom, 0.0), 1.0)
    return f"rgb(240, {240 * (1 - normalized_val)}, {240 * (1 - normalized_val)})"


def basic_router_vis(text, layer, max_val=None, min_val=None):
    """
    text: The text to visualize
    layer: The layer index
    max_val: The top end of our probability range, defaults to the maximum probability
    min_val: The bottom end of our probability range, defaults to the minimum probability

    Returns a string of HTML that displays a token-by-expert router probability heatmap.
    """
    try:
        probs, str_tokens = get_router_probs(text, layer)
    except Exception as error:
        return str(error)

    prob_max = probs.max()
    prob_min = probs.min()
    if max_val is None:
        max_val = prob_max
    if min_val is None:
        min_val = prob_min

    htmls = [style_string]
    htmls.append(f"<h4>Layer: <b>{int(layer)}</b></h4>")
    htmls.append(
        f"<h4>Display Range: <b>{max_val:.4f}</b>. Min Range: <b>{min_val:.4f}</b></h4>"
    )
    htmls.append(
        f"<h4>Actual Probabilities: <b>{prob_min:.4f}</b> to <b>{prob_max:.4f}</b></h4>"
    )

    htmls.append("<div class='router-scroll'><table class='router-heatmap'>")
    htmls.append("<tr><th>Expert</th>")
    for tok in str_tokens:
        safe_tok = html.escape(tok)
        htmls.append(f"<th class='token-header'>{safe_tok}</th>")
    htmls.append("</tr>")

    num_experts = probs.shape[1]
    for expert_idx in range(num_experts):
        htmls.append(f"<tr><th class='expert-label'>E{expert_idx}</th>")
        for token_idx in range(len(str_tokens)):
            prob = probs[token_idx, expert_idx]
            color = calculate_color(prob, max_val, min_val)
            htmls.append(
                f"<td style='background-color:{color}' title='p={prob:.6f}'>{prob:.4f}</td>"
            )
        htmls.append("</tr>")

    htmls.append("</table></div>")
    return "".join(htmls)


In [11]:
# NBVAL_IGNORE_OUTPUT
# The function outputs a string of HTML
default_max_val = 1.0
default_min_val = 0.0
default_html_string = basic_router_vis(
    default_text,
    default_layer,
    max_val=default_max_val,
    min_val=default_min_val,
)

# IPython lets us display HTML
print("Displayed HTML")
display(HTML(default_html_string))

# We can also print the string directly
print("HTML String - it's just raw HTML code!")
print(default_html_string)


Displayed HTML


Expert,Sunday
E0,0.0119
E1,0.0244
E2,0.0403
E3,0.0068
E4,0.0213
E5,0.0106
E6,0.0043
E7,0.0094
E8,0.0052
E9,0.0095


HTML String - it's just raw HTML code!
<style>
    table.router-heatmap {
        border-collapse: collapse;
        font-family: monospace;
        font-size: 12px;
    }
    table.router-heatmap th, table.router-heatmap td {
        border: 1px solid rgb(123, 123, 123);
        padding: 4px;
        text-align: center;
        min-width: 48px;
    }
    table.router-heatmap th.token-header {
        writing-mode: vertical-rl;
        transform: rotate(180deg);
        white-space: pre-wrap;
    }
    table.router-heatmap th.expert-label {
        position: sticky;
        left: 0;
        background: white;
    }
    div.router-scroll {
        overflow-x: auto;
        max-width: 100%;
    }
    </style><h4>Layer: <b>7</b></h4><h4>Display Range: <b>1.0000</b>. Min Range: <b>0.0000</b></h4><h4>Actual Probabilities: <b>0.0026</b> to <b>0.1014</b></h4><div class='router-scroll'><table class='router-heatmap'><tr><th>Expert</th><th class='token-header'>Sunday</th></tr><tr><th class='expe

## Create Interactive UI

We now put all these together to create an interactive visualization in Gradio.

The interface now shows all experts for one selected layer at a time as a token-by-expert heatmap.

In [12]:
# The `with gr.Blocks() as demo:` syntax just creates a variable called demo containing all these components
with gr.Blocks() as demo:
    gr.HTML(value=f"Hacky Interactive OLMoE Router Heatmap for {model_name}")
    # The input elements
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(label="Text", value=default_text)
            # Precision=0 makes it an int, otherwise it's a float
            # Value sets the initial default value
            layer = gr.Number(label="Layer", value=default_layer, precision=0)
            # If empty, these two map to None
            max_val = gr.Number(label="Max Value", value=default_max_val)
            min_val = gr.Number(label="Min Value", value=default_min_val)
            inputs = [text, layer, max_val, min_val]
        with gr.Column():
            # The output element
            out = gr.HTML(label="Router Probability Heatmap", value=default_html_string)
    for inp in inputs:
        inp.change(basic_router_vis, inputs, out)


We can now launch our demo element, and we're done! The setting share=True even gives you a public link to the demo (though it just redirects to the backend run by this notebook, and will go away once you turn the notebook off!). Sharing makes it much slower, and can be turned off if you aren't in Colab.

**Exercise:** Explore how the full expert distribution changes across layers and across different texts. Which tokens concentrate mass on a small set of experts, and which spread it across many experts?

In [13]:
# NBVAL_IGNORE_OUTPUT
demo.launch(share=True, height=1000)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f65233e958c95a4abe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
